# Pandas' Function application

**Estimated Time**: 25 minutes.

In [1]:
import pandas as pd
import numpy as np

Functions can be applied:

- to each row or column (row/column-wise)
- to each individual element (element-wise)

Or it may be applied to the data structure as a whole, whether `DataFrame` or `Series`.

### 1. Tablewise Function Application: [`pipe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html#pandas.DataFrame.pipe)

A pipeline is a sequence of data processing steps. In Pandas:

- input: `DataFrame`, output: `DataFrame`
- input: `Series`, output: `Series`

Example:

In [14]:
def extract_city_name(df):
    """
    Chicago, IL -> Chicago for city_name column
    """
    df["city_name"] = df["city_and_code"].str.split(",").str.get(0)
    return df

def add_country_name(df, country_name=None):
    """
    Chicago -> Chicago-US for city_name column
    """
    col = "city_name"
    df["city_and_country"] = df[col] + country_name
    return df

df_p = pd.DataFrame({"city_and_code": ["Chicago, IL"]})

Regular function calling works, but it can get messy:

In [15]:
add_country_name(extract_city_name(df_p), country_name="US")

,city_and_code,city_name,city_and_country
0,"Chicago, IL",Chicago,ChicagoUS


The issue with nested function calls is that they can be hard to read. For example, consider the following code (5 steps):

```python
result = format_currency(
    calculate_tax(
        add_country_name(
            extract_city_name(
                clean_date_format(df_p)
            ), 
            country_name="US"
        ), 
        tax_rate=0.08
    )
)
```

The issue with this **inside-out** is:

- Can you easily see the order of operations? (`clean_date_format -> extract_city_name -> ...`)
- Can you easily add or remove steps?
- Can you easily disable just one step for debugging?
- Can you easily tell which parameters go with which function?

Using the `pipe()` you can keep the logic linear (top-to-bottom) and improve readability:

```python
result = (df_p
    .pipe(clean_date_format)
    .pipe(extract_city_name)
    .pipe(add_country_name, country_name="US")
    .pipe(calculate_tax, tax_rate=0.08)
    .pipe(format_currency)
)
```
> Note: The parentheses around the entire expression are necessary to allow line breaks.

Advantages of using `pipe()`:

- The order is clear (top-to-bottom).
- You can easily add/remove steps.
- You can easily disable just one step for debugging.
- You can easily tell which parameters go with which function.

Let's apply that to our example functions:

In [18]:
(
    df_p
    .pipe(extract_city_name)
    .pipe(add_country_name, country_name="US")
)

,city_and_code,city_name,city_and_country
0,"Chicago, IL",Chicago,ChicagoUS


### 2. Row or Column-wise Function Application: [`apply(func, axis=0)`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html#pandas.DataFrame.apply)

`axis`: Axis along which the function is applied:

- `0` or `'index'`: apply function to each column.
- `1` or `'columns'`: apply function to each row.

let’s create some example objects:

In [5]:
index = pd.date_range("1/1/2000", periods=8)
s = pd.Series(np.random.randn(5), index=["a", "b", "c", "d", "e"])
df = pd.DataFrame(np.random.randn(8, 3), index=index, columns=["A", "B", "C"])

In [6]:
def custom_mean(x: pd.Series):
    return np.mean(x)

In [7]:
df.apply(custom_mean, axis=0) # column-wise mean

A   -0.266987
B    0.693155
C   -0.108896
dtype: float64

In [20]:
df.apply(custom_mean, axis=1) # row-wise mean

2000-01-01    0.007308
2000-01-02    0.081878
2000-01-03    0.654347
2000-01-04   -0.103659
2000-01-05    0.622983
2000-01-06    0.351335
2000-01-07   -1.180132
2000-01-08    0.412000
Freq: D, dtype: float64

Using a broadcastable operation (like addition, multiplication, etc.) means the axis parameter doesn't matter:

- Example: `axis=0` (column-wise) or `axis=1` (row-wise) when doing `x + 10`

In [9]:
def my_square(x: pd.Series):
    return x ** 2

# Equal results whether applied column-wise or row-wise
df.apply(my_square, axis=0) == df.apply(my_square, axis=1)

,A,B,C
2000-01-01,True,True,True
2000-01-02,True,True,True
2000-01-03,True,True,True
2000-01-04,True,True,True
2000-01-05,True,True,True
2000-01-06,True,True,True
2000-01-07,True,True,True
2000-01-08,True,True,True


In [10]:
# Using lambda functions
df.apply(lambda x: np.mean(x), axis=1) # row-wise mean

2000-01-01    0.007308
2000-01-02    0.081878
2000-01-03    0.654347
2000-01-04   -0.103659
2000-01-05    0.622983
2000-01-06    0.351335
2000-01-07   -1.180132
2000-01-08    0.412000
Freq: D, dtype: float64

### 3. Applying Elementwise Functions: [`map()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html#pandas.DataFrame.map)

The method `map()`:

- Accepts any Python function taking a single value and returning a single value
- Works for both `Series` and `DataFrame`

For example:

In [11]:
def f(x: pd.DataFrame | pd.Series):
    return x + 10

In [12]:
df.map(f)

,A,B,C
2000-01-01,8.638244,11.163979,10.219700
2000-01-02,10.133810,10.026453,10.085371
2000-01-03,11.318173,11.043922,9.600946
2000-01-04,9.130167,12.729241,7.829615
2000-01-05,10.836756,10.624347,10.407846
2000-01-06,9.986248,10.067742,11.000017
2000-01-07,8.172735,9.811244,8.475623
2000-01-08,9.647971,10.078311,11.509717


In [13]:
df['A'].map(f)

2000-01-01     8.638244
2000-01-02    10.133810
2000-01-03    11.318173
2000-01-04     9.130167
2000-01-05    10.836756
2000-01-06     9.986248
2000-01-07     8.172735
2000-01-08     9.647971
Freq: D, Name: A, dtype: float64

---

References:

- [Pandas' User Guide > Essential basic functionality > Function application](https://pandas.pydata.org/docs/user_guide/basics.html#function-application)